# Using commit vith onnx quantized

In [ ]:
from utils_dino_final_pylib import *
import onnxruntime
import time

def get_onnx_mask_from_bboxes_onnx(bboxes, ort_session):

    all_masks = []

    for bbox in bboxes:

        onnx_box_coords = bbox.reshape(2, 2)
        onnx_box_labels = np.array([2,3])

        onnx_coord = np.concatenate([ onnx_box_coords], axis=0)[None, :, :]
        onnx_label = np.concatenate([ onnx_box_labels], axis=0)[None, :].astype(np.float32)

        # print("onnx_coord before", onnx_coord)


        onnx_coord = predictor.transform.apply_coords(onnx_coord, image.shape[:2]).astype(np.float32)
        
        # print("onnx_coord after", onnx_coord)
        # print("onnx_label", onnx_label)
        # print("onnx_label.shape", onnx_label.shape)

        # print(image.shape[:2])

        onnx_mask_input = np.zeros((1, 1, 256, 256), dtype=np.float32)
        onnx_has_mask_input = np.zeros(1, dtype=np.float32)


        ort_inputs = {
            "image_embeddings": image_embedding,
            "point_coords": onnx_coord,
            "point_labels": onnx_label,
            "mask_input": onnx_mask_input,
            "has_mask_input": onnx_has_mask_input,
            "orig_im_size": np.array(image.shape[:2], dtype=np.float32)
        }

        masks, _, _ = ort_session.run(None, ort_inputs)
        masks = masks > predictor.model.mask_threshold
        # print("masks.shape", masks.shape)

        all_masks.append(masks)

    return all_masks

In [ ]:
def get_door_bboxes_points(SOURCE_IMAGE_PATH):

    # 1) DEFALUT DOOR PARAMS + 80% area check = OK !!!
    # BOX_THRESHOLD   = 0.65
    # TEXT_THRESHOLD  = 0.20
    # NMS_THRESHOLD   = 0.8
    # TEXT_PROMPT = "doors"

    # 2) NEW DOOR PARAMS + 80% area check = OK !!!
    BOX_THRESHOLD   = 0.60
    TEXT_THRESHOLD  = 0.80
    NMS_THRESHOLD   = 0.8
    TEXT_PROMPT = "doors"

    if os.path.exists(SOURCE_IMAGE_PATH):
        
        ######################### DO GROUNDING DETECTION ##########################
        # load image
        image = cv2.imread(SOURCE_IMAGE_PATH)

        image_source, image_dino = load_image(SOURCE_IMAGE_PATH)
        boxes, logits, phrases = predict(
            model=grounding_dino_model,
            image=image_dino,
            caption=TEXT_PROMPT,
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD
        )

        h, w, _ = image_source.shape
        boxes = boxes * torch.Tensor([w, h, w, h])
        xyxy = box_convert(boxes=boxes, in_fmt="cxcywh", out_fmt="xyxy").numpy()

        detections_box = xyxy
        detections_scores = logits.cpu().numpy()
        
        ############################################################################
        ######################### DO NMS POSTPROCESSING ##########################
        # NMS post process
        nms_idx = nms(
            torch.from_numpy(detections_box), 
            torch.from_numpy(detections_scores), 
            NMS_THRESHOLD
        ).numpy().tolist()


        detections_box = detections_box[nms_idx]
        detections_scores = detections_scores[nms_idx]

        if len(detections_box) == 0:
            print("No door detected")
            return [], [], []

        #############################################################################
        ######################### DO SAM SEGMENTATION ###############################

        # Prompting SAM with detected boxes
        def segment(sam_predictor: SamPredictor, image: np.ndarray, xyxy: np.ndarray) -> np.ndarray:
            sam_predictor.set_image(image)
            result_masks = []
            for box in xyxy:
                masks, scores, logits = sam_predictor.predict(
                    box=box,
                    multimask_output=True
                )
                index = np.argmax(scores)
                result_masks.append(masks[index])
            return np.array(result_masks)


        # convert detections to masks
        detections_masks = segment(
            sam_predictor=sam_predictor,
            image=cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
            xyxy=detections_box
        )

        filtered_masks, filtered_boxes, filtered_scores = detections_masks, detections_box, detections_scores
        # filtered_boxes, filtered_scores = detections_box, detections_scores

        ################################################################################################
        # FILTER MASKS #
        ################################################################################################
        # First remove small masks (< 10% image area)
        # filtered_masks, filtered_boxes, filtered_scores = filter_small_masks(
        #                                                     masks=detections_masks,
        #                                                     boxes=detections_box,
        #                                                     scores=detections_scores, 
        #                                                     image_shape=image.shape[:2],  # (H, W)
        #                                                     min_area_ratio=0.01
        #                                                 )

        # filtered_masks, filtered_boxes, filtered_scores = filter_noisy_masks(filtered_masks,
        #                                                     filtered_boxes,
        #                                                     scores=filtered_scores, 
        #                                                     max_components=20,
        #                                                     min_component_area=10)


        # filtered_masks, filtered_boxes, filtered_scores = remove_masks_that_contain_others(
        #                                                             masks=filtered_masks,
        #                                                             boxes=filtered_boxes,
        #                                                             scores=filtered_scores,
        #                                                             threshold=0.9
        #                                                             )
        ################################################################################################
        ################################################################################################

        selected_points = []
        for bbox in filtered_boxes:  # each bbox = [x_min, y_min, x_max, y_max]
            x_min, y_min, x_max, y_max = bbox

            # Center of bbox
            cx = int((x_min + x_max) / 2)
            cy = int((y_min + y_max) / 2)

            selected_points.append([cx, cy])

        ################################################################################################
        ################################################################################################
        print("DOOR SCORES", filtered_scores)
        # return filtered_boxes
        # return filtered_boxes, selected_points
        return filtered_masks, filtered_boxes, selected_points
    

def get_wall_bboxes_points(SOURCE_IMAGE_PATH):

    BOX_THRESHOLD = 0.25
    TEXT_THRESHOLD = 0.25
    NMS_THRESHOLD = 0.8

    TEXT_PROMPT = "walls"

    print("Getting wall mask from ", SOURCE_IMAGE_PATH)

    if os.path.exists(SOURCE_IMAGE_PATH):
        
        ######################### DO GROUNDING DETECTION ##########################
        # load image
        image = cv2.imread(SOURCE_IMAGE_PATH)

        image_source, image_dino = load_image(SOURCE_IMAGE_PATH)
        boxes, logits, phrases = predict(
            model=grounding_dino_model,
            image=image_dino,
            caption=TEXT_PROMPT,
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD
        )

        h, w, _ = image_source.shape
        boxes = boxes * torch.Tensor([w, h, w, h])
        xyxy = box_convert(boxes=boxes, in_fmt="cxcywh", out_fmt="xyxy").numpy()

        detections_box = xyxy
        detections_scores = logits.cpu().numpy()

        ############################################################################
        ######################### DO NMS POSTPROCESSING ##########################
        # NMS post process
        nms_idx = nms(
            torch.from_numpy(detections_box), 
            torch.from_numpy(detections_scores), 
            NMS_THRESHOLD
        ).numpy().tolist()


        detections_box = detections_box[nms_idx]
        detections_scores = detections_scores[nms_idx]

        #############################################################################
        ######################### DO SAM SEGMENTATION ###############################

        # Prompting SAM with detected boxes
        def segment(sam_predictor: SamPredictor, image: np.ndarray, xyxy: np.ndarray) -> np.ndarray:
            sam_predictor.set_image(image)
            result_masks = []
            for box in xyxy:
                masks, scores, logits = sam_predictor.predict(
                    box=box,
                    multimask_output=True
                )
                index = np.argmax(scores)
                result_masks.append(masks[index])
            return np.array(result_masks)


        # convert detections to masks
        detections_masks = segment(
            sam_predictor=sam_predictor,
            image=cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
            xyxy=detections_box
        )
        filtered_masks, filtered_boxes, filtered_scores = detections_masks, detections_box, detections_scores
        ################################################################################################
        # FILTER MASKS #
        ################################################################################################
        # First remove small masks (< 10% image area)
        filtered_masks, filtered_boxes, filtered_scores = filter_small_masks(
                                                            masks=detections_masks,
                                                            boxes=detections_box,
                                                            scores=detections_scores, 
                                                            image_shape=image.shape[:2],  # (H, W)
                                                            min_area_ratio=0.01
                                                        )

        filtered_masks, filtered_boxes, filtered_scores = filter_noisy_masks(filtered_masks,
                                                            filtered_boxes,
                                                            scores=filtered_scores, 
                                                            max_components=20,
                                                            min_component_area=10)


        filtered_masks, filtered_boxes, filtered_scores = remove_masks_that_contain_others(
                                                                    masks=filtered_masks,
                                                                    boxes=filtered_boxes,
                                                                    scores=filtered_scores,
                                                                    threshold=0.9
                                                                    )

        ################################################################################################
        ################################################################################################
        selected_points = []
        for i, mask in enumerate(filtered_masks):  # mask: (H, W) binary
            mask_uint8 = mask.astype(np.uint8)

            # Compute center from moments
            moments = cv2.moments(mask_uint8)
            if moments["m00"] != 0:
                original_cx = int(moments["m10"] / moments["m00"])
                original_cy = int(moments["m01"] / moments["m00"])
            else:
                ys, xs = np.where(mask)
                original_cx = int(np.mean(xs))
                original_cy = int(np.mean(ys))

            cx, cy = original_cx, original_cy

            if not mask[cy, cx]:
                ys, xs = np.where(mask)
                distances = np.sqrt((xs - cx)**2 + (ys - cy)**2)

                # Get index of 25th percentile closest point
                percentile_index = int(len(distances) * 0.2)
                sorted_indices = np.argsort(distances)

                idx = sorted_indices[percentile_index]
                cx, cy = int(xs[idx]), int(ys[idx])

            selected_points.append([cx, cy])

        ################################################################################################
        ################################################################################################
        print("WALL SCORES", filtered_scores)

        return filtered_masks, filtered_boxes, selected_points

In [ ]:
def mask_iou(mask1, mask2):
    inter = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return inter / union if union > 0 else 0

def filter_doors_by_mask(door_masks, door_bboxes, wall_masks, iou_thresh=0.95):
    filtered_doors_masks = []
    filtered_doors_bboxes = []
    for dmask, dbox in zip(door_masks, door_bboxes):
        keep = True
        for wmask in wall_masks:
            if mask_iou(dmask, wmask) > iou_thresh:
                keep = False
                break
        if keep:
            filtered_doors_masks.append(dmask)
            filtered_doors_bboxes.append(dbox)
    return filtered_doors_masks, filtered_doors_bboxes

In [ ]:
start_time = time.time()

# FOLDER_PATH = r"D:\3d-recon\Grounded-Segment-Anything\with_door_dataset"
# FOLDER_PATH = r"D:\3d-recon\Grounded-Segment-Anything\door_dataset"
# FOLDER_PATH = r"D:\3d-recon\datasets\ASARoomImage"
FOLDER_PATH = r"D:\3d-recon\RoomSceneSegmentation\RoomSceneImage"

# OUTPUT_FOLDER_PATH = r"D:\3d-recon\Grounded-Segment-Anything\with_door_output"
# OUTPUT_FOLDER_PATH = r"D:\3d-recon\Grounded-Segment-Anything\door_dataset_output"
# OUTPUT_FOLDER_PATH = r"D:\3d-recon\Grounded-Segment-Anything\asa_door_output"
OUTPUT_FOLDER_PATH = r"D:\3d-recon\Grounded-Segment-Anything\roomscene_door_output"

for IMAGE_NAME in os.listdir(FOLDER_PATH):

  SOURCE_IMAGE_PATH = os.path.join(FOLDER_PATH, IMAGE_NAME)

  print(f"================ Processing image: {IMAGE_NAME} ================")

  wall_masks, wall_bboxes, wall_selected_points = get_wall_bboxes_points(SOURCE_IMAGE_PATH)

  (floor_bboxes, 
  floor_selected_points, 
  rug_points) = get_floor_bboxes_points_with_rug(SOURCE_IMAGE_PATH)

  door_masks, door_bboxes, door_selected_points = get_door_bboxes_points(SOURCE_IMAGE_PATH)
  print("door_bboxes", door_bboxes)
  print("door_selected_points", door_selected_points)


  output = {
    'wall_bboxes': wall_bboxes.tolist(),
    'wall_selected_points':wall_selected_points,
    'floor_bboxes':floor_bboxes.tolist(),
    'floor_selected_points': floor_selected_points,
    'rug_points': rug_points,
    'door_bboxes': np.array(door_bboxes).tolist(),
    'door_selected_points': door_selected_points
        }
  


  import json
  # Save to JSON
  with open(os.path.join(OUTPUT_FOLDER_PATH, f"auto_detect_{IMAGE_NAME}.json"), "w") as f:
      json.dump(output, f, indent=4)  # indent for readability


  image = cv2.imread(SOURCE_IMAGE_PATH)
  image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(image.shape)

  # if len(door_bboxes) != 0:
    # checkpoint = r"./sam_vit_h_4b8939.pth"

    # model_type = "vit_h"
    # sam = sam_model_registry[model_type](checkpoint=checkpoint)


    # sam.to(device='cuda')
    # predictor = SamPredictor(sam)
    # predictor.set_image(image)
    # image_embedding = predictor.get_image_embedding().cpu().numpy()

    # np.save("image_embedding_vith.npy", image_embedding)

    # print("DONE SAM EMBEDDING", time.time()-start_time)

    # onnx_model_path = r"D:\3d-recon\Grounded-Segment-Anything\sam_quantized.onnx"
    # ort_session = onnxruntime.InferenceSession(onnx_model_path)
    # door_masks = get_onnx_mask_from_bboxes_onnx(door_bboxes, ort_session)


  plt.figure()
  plt.imshow(image)
    # plt.imshow(image)
  for idx, (mask, bbox, point) in enumerate(zip(door_masks, door_bboxes, np.array(door_selected_points))):
    show_box(bbox, plt.gca())
    show_point(point, plt.gca())
    show_mask(mask, plt.gca(),)

  plt.axis('off')
  # plt.savefig(os.path.join(OUTPUT_FOLDER_PATH, f"{IMAGE_NAME}.jpg"))
  plt.show()

  # plt.figure()
  # plt.imshow(image)

  # for idx, (mask, bbox, point) in enumerate(zip(wall_masks, wall_bboxes, np.array(wall_selected_points))):
  #   print(mask.shape)
  #   show_mask(mask, plt.gca())
  #   show_box(bbox, plt.gca())
  
  # plt.axis('off')
  # # plt.savefig(os.path.join(OUTPUT_FOLDER_PATH,f"{IMAGE_NAME}.jpg"))
  # plt.show()

  # else:
  #   plt.figure()
  #   plt.imshow(image)
  #   plt.axis('off')
  #   plt.savefig(os.path.join(OUTPUT_FOLDER_PATH,f"{IMAGE_NAME}.jpg"))
  #   plt.show()


  def is_fully_visible(door_mask, door_bbox, threshold=0.8):
      x1, y1, x2, y2 = map(int, door_bbox)  # convert bbox coords to int
      bbox_area = (x2 - x1) * (y2 - y1)
      
      # Mask area inside bbox
      mask_area = np.sum(door_mask[y1:y2, x1:x2] > 0)
      
      visible_ratio = mask_area / bbox_area
      return visible_ratio >= threshold
  
  filtered_masks = []
  filtered_bboxes = []
  filtered_points = []

  for mask, bbox, point in zip(door_masks, door_bboxes, door_selected_points):
      
      # print(mask.shape, mask)
      if is_fully_visible(mask, bbox, threshold=0.8):
          filtered_masks.append(mask)
          filtered_bboxes.append(bbox)
          filtered_points.append(point)

  plt.figure()
  plt.imshow(image)
    # plt.imshow(image)
  for idx, (mask, bbox, point) in enumerate(zip(filtered_masks, filtered_bboxes, np.array(filtered_points))):
    show_box(bbox, plt.gca())
    show_point(point, plt.gca())
    show_mask(mask, plt.gca(),)

  plt.axis('off')
  plt.savefig(os.path.join(OUTPUT_FOLDER_PATH, f"{IMAGE_NAME}.jpg"))
  plt.show()
